In [2]:
import os

from dotenv import load_dotenv
from google import genai

# This reads your .env file and loads the variables into your system
load_dotenv() 

# The Gemini SDK automatically looks for the "GEMINI_API_KEY" variable behind the scenes!
client = genai.Client()

Choosing a model
Sentence-transformers supports many models. The right one depends on your task, your language, and the resources you have. Larger models are usually slower, so for our FAQ dataset of short English texts a small model is enough. Try a few on your own data and keep the one that works best.

We'll use all-MiniLM-L6-v2:

384-dimensional vectors (compact)
Fast on CPU
Good quality for general English text
Uses cosine similarity (we'll explain this below)


In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

start with a query:

In [4]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

v1 is a vector, an array of 384 numbers. Each number stands for some concept the model learned. We can't read off what any one of them means. But two vectors with similar values point to texts about similar things.

In [5]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [6]:
v1.dot(dv)

np.float32(0.32332397)

Now we try an unrelated query:

In [7]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

This time the similarity with the document should be much smaller:

In [8]:
v2.dot(dv)

np.float32(0.019730438)

Cosine similarity
The all-MiniLM-L6-v2 model outputs normalized vectors - vectors with unit length. When both vectors are normalized, the dot product equals cosine similarity. That's why the model documentation says it "uses cosine similarity."

Cosine similarity measures the angle between two vectors, ignoring their length:

1.0 = same direction (similar)
0.0 = perpendicular (unrelated)
-1.0 = opposite direction (opposite meaning)
Formally, if theta is the angle between two vectors, cosine similarity is cos(theta):

cos(0) = 1 - vectors point in the same direction
cos(90) = 0 - vectors are perpendicular
cos(180) = -1 - vectors point in opposite directions
Because our vectors are normalized, the dot product gives us cosine similarity directly. This is why we can use v1.dot(dv) to compare texts.

In practice, we rarely get cosine similarity below 0. The embedding model maps text to a region of the vector space where most vectors have positive components. There's no concept of "opposite meaning" that maps to a vector pointing the other way.

Embedding Our Dataset

In [9]:
# We use it here:
from ingest import load_faq_data

documents = load_faq_data()

Generating embeddings

In [10]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

Now we generate the embeddings. We have about 1200 texts here. We won't hand the model all of them at once. That takes a long time, and we can't see what's happening inside. Instead we split them into batches.

In [11]:
from tqdm.auto import tqdm

In [12]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

We end up with 1208 vectors. On a GPU this is fast. Most of us run on Codespaces without a GPU, so it takes a bit, but it's a one-off.

We turn them into a 2-dimensional array (matrix) where

rows are documents (vectors)
columns are dimensions of the vectors

In [13]:
import numpy as np
X = np.array(vectors)

In [14]:
print(X)

[[-0.02670622 -0.12245756  0.01594418 ... -0.00230645 -0.11218391
  -0.02365551]
 [-0.01099553 -0.11074745 -0.02536936 ...  0.09022227 -0.02697366
   0.01975676]
 [-0.08896549 -0.06128184  0.00775605 ...  0.0405971   0.0047928
  -0.0274594 ]
 ...
 [-0.03652925  0.01415433 -0.06838643 ...  0.04316785  0.08105534
  -0.02148626]
 [-0.13091588 -0.06990597 -0.00931879 ... -0.00044345 -0.01285729
   0.01426925]
 [-0.07984771  0.01926989  0.02544983 ... -0.03368025 -0.01884021
   0.05837058]]


Number of documents vs number of dimensions.

In [15]:
X.shape

(1350, 384)

Vector Search

In [16]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [17]:
scores = X.dot(v_query)

This is matrix-vector multiplication. Each element i of scores is the cosine similarity between document i (row i of X) and v_query.

We could compute the same thing with a for loop:

In [18]:
scores = [v_query.dot(X[i]) for i in range(len(X))]

Best match

In [19]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.76294106))

In [20]:
documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [21]:
{"id": "3f1424af17",
 "course": "data-engineering-zoomcamp",
 "section": "General Course-Related Questions",
 "question": "Course: Can I still join the course after the start date?",
 "answer": "Yes, even if you don't register, you're still eligible..."}

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible..."}

Top 5 results

Usually we want more than the single best match, so let's pull the top 5.

np.argsort sorts from lowest to highest, so the last 5 are the top ones

In [22]:
top5 = np.argsort(scores)[-5:]

In [23]:
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [24]:
# Convert to array, negate it, sort it, and slice the first 5
top5_indices = np.argsort(-np.array(scores))[:5]

# Read off the winning scores
np.array(scores)[top5_indices]

array([0.76294106, 0.7579372 , 0.7192131 , 0.6536312 , 0.56009996],
      dtype=float32)

np.array(scores) translates your standard list into a NumPy math array.

- instantly flips all the positive numbers to negative, turning your highest scores into the smallest numbers.

np.argsort() sorts those new numbers from smallest to largest.

[:5] slices off just the first 5 results.

In [25]:
top5 = np.argsort(-np.array(scores))[:5]

In [26]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.76294106
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579372
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192131
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions'

Vector Search with minsearch

In the previous section we did vector search by hand with numpy. We embedded the query, computed dot products, and found the best matches. Writing the argsort and matrix code every time gets old, and it can't filter by course. So instead we'll use a library that wraps all of it.

Both classes share the same API:

fit to index data
search to query
filter_dict in search to filter by keyword
It's the simplest way to get started with vector search.

Creating the index

In [27]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

Searching

In [28]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [29]:
results[0]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

In [30]:
{"id": "74eb249bbf",
 "course": "llm-zoomcamp",
 "section": "General Course-Related Questions",
 "question": "I just discovered the course. Can I still join?",
 "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions."}

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

Filtering by course

Like the text index, we can filter by keyword fields. This matters for user experience. A student in LLM Zoom Camp doesn't care about answers from the data engineering course. So we narrow to their course first, then score only within it.

In [31]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

Now that we can run vector search, let's use it in RAG.

RAG with Vector Search

In module 1, we built a RAG pipeline with three steps:

def rag(question):
    search_results = search(question) // Retrieval
    user_prompt = build_prompt(question, search_results) // Augmentation
    return llm(user_prompt) // Generation

In [32]:
#from dotenv import load_dotenv
#from google import genai

# This reads your .env file and loads the variables into your system
#load_dotenv() 

# The Gemini SDK automatically looks for the "GEMINI_API_KEY" variable behind the scenes!
#client = genai.Client()

from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [33]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=client,
)

In [34]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, you can still join. However, if you wish to receive a certificate, you will need to submit your project while submissions are still being accepted.'

In [35]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

Using it

In [37]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=client,
)

In [38]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

Vector Search with sqlitesearch

sqlitesearch
sqlitesearch is the persistent sibling of minsearch, and it solves both problems at once.

We already used it in module 1 for persistent text search. It also does vector search through its VectorSearchIndex class. It stores vectors in SQLite, a real on-disk database, and uses ANN strategies for retrieval. Because the data lives on disk, one process can write the vectors and another can read them back.

If you didn't install it in the previous module, add it to your project:

Creating the index

In [39]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

sqlitesearch supports three ANN modes:

lsh (default): up to 100K vectors, random hyperplane projections
ivf: 10K-500K vectors, K-means clustering
hnsw: 10K-1M+ vectors, proximity graph (highest recall)
For our small dataset, lsh is fine. All modes use two-phase search: approximate candidate retrieval, then exact cosine similarity reranking.



Indexing the data

In [40]:
vs_index.fit(vectors, documents)

Searching

Search works the same way as with minsearch. We always encode the query into a vector first. This is one thing that makes vector search heavier than text search. With text search we'd throw the raw query straight at the engine.

In [41]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [42]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'doc_id': '41aabbd7c5'},
 {'course': 'mlops-zoomcamp',
  'section': 'General Cour

Filtering by course

In [43]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

Closing the connection

In [44]:
vs_index.close()

Reopening the index

In [45]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [46]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)

Using sqlitesearch vector search in RAG

In [47]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [49]:
#from rag_helper import RAGBase
#from dotenv import load_dotenv
#from openai import OpenAI

#load_dotenv()
#openai_client = OpenAI()

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=client,
)

In [50]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [51]:
vs_index.close()

Comparing minsearch and sqlitesearch for vector search
Here is how the two compare:

minsearch VectorSearch: in-memory (numpy), exact cosine similarity, must re-compute embeddings on startup, good for experiments and notebooks
sqlitesearch VectorSearchIndex: persistent (SQLite .db file), ANN (LSH/IVF/HNSW) with exact rerank, can open an existing index, good for projects and persistence
This is probably the last you'll hear of sqlitesearch. I built it for teaching, to show the ingestion-then-deployment split.

It does have a real use though. Its only dependencies are SQLite and numpy. So it runs on any host that offers a free SQLite database, where a dedicated vector database would cost extra. For most work you'll reach for something else, which is what we do next.

Vector Search with PGVector

Many real databases can do vector search. Elasticsearch has it, and there are dedicated stores like Qdrant and Chroma. We'll go with Postgres. Most of us already run it at work, and the data engineering course uses it too. The concept is the same as with sqlitesearch, only the database under the hood changes.

pgvector is the PostgreSQL extension that makes this work. Install it and Postgres can do vector similarity search. On top of that you get the usual production features, like concurrent access, transactions, and large datasets.

We'll run Postgres with pgvector in Docker.